Connecting to Bigquery Data

In [ ]:
import pandas as pd

In [ ]:
from google.colab import auth
auth.authenticate_user()

In [ ]:
PROJECT_ID = "project-47cd8d9d-29e8-45d0-a63"

from google.cloud import bigquery
client = bigquery.Client(project=PROJECT_ID)

In [ ]:
members_query = """
SELECT *
FROM `medicare_claims.dim_members`
"""
df_members = client.query(members_query).to_dataframe()

In [ ]:
claims_query = """
SELECT *
FROM `medicare_claims.fact_claims`
"""
df_claims = client.query(claims_query).to_dataframe()

In [ ]:
print(df_members.head())

          member_id  member_birth_date  gender  has_diabetes  \
0  09A323B4746F5452           19090101  Female             1   
1  F067303F9580A140           19090101  Female             1   
2  02C598F9B4F67D40           19090101  Female             1   
3  C0C1D7D054E71722           19090101    Male             1   
4  BBE8F3FF8DA00F17           19090101  Female             1   

   has_kidney_disease  has_heart_failure member_health_profile  
0                   1                  1      All 3 Conditions  
1                   1                  1      All 3 Conditions  
2                   1                  1      All 3 Conditions  
3                   1                  1      All 3 Conditions  
4                   1                  1      All 3 Conditions  


In [ ]:
print(df_claims.head())

          member_id  admission_date  discharge_date  plan_paid_amount  \
0  1DBED23FBABFF6C6        20071127        20080101            8000.0   
1  35F495900193DFA9        20071128        20080102           57000.0   
2  81E7E08EDC17B3B7        20071204        20080106            5000.0   
3  804C869EDCCA7059        20071205        20080109           15000.0   
4  95F29932C3D13B6A        20071206        20080110           30000.0   

   primary_payer_paid_amount  
0                        0.0  
1                        0.0  
2                        0.0  
3                        0.0  
4                        0.0  


In [ ]:
print(df_members.shape)
print(df_claims.shape)
print(df_members.describe())
print(df_claims.describe())

(116352, 7)
(66773, 5)
       member_birth_date  has_diabetes  has_kidney_disease  has_heart_failure
count           116352.0      116352.0            116352.0           116352.0
mean     19364180.585224      0.378678            0.160599           0.284954
std        125044.606968       0.48506            0.367162           0.451395
min           19090101.0           0.0                 0.0                0.0
25%           19281101.0           0.0                 0.0                0.0
50%           19360501.0           0.0                 0.0                0.0
75%           19420301.0           1.0                 0.0                1.0
max           19831201.0           1.0                 1.0                1.0
       admission_date   discharge_date  plan_paid_amount  \
count         66773.0          66773.0      66773.000000   
mean   20088463.03765  20088611.518847       9573.632756   
std         7587.0811      7558.797069       9315.073232   
min        20071127.0       2008010

In [ ]:
df_claims[df_claims['plan_paid_amount'] < 0].count()

,0
member_id,55
admission_date,55
discharge_date,55
plan_paid_amount,55
primary_payer_paid_amount,55


In [ ]:
df_claims[df_claims['plan_paid_amount'] < 0]

,member_id,admission_date,discharge_date,plan_paid_amount,primary_payer_paid_amount
933,5E45042ECCBF8AF4,20080116,20080131,-50.0,0.0
3917,C8C35DAF0F9A864B,20080305,20080318,-200.0,0.0
4252,8934A2FAEDD4A39B,20080311,20080312,-800.0,0.0
6025,7C2822297CEAC5E6,20080403,20080405,-200.0,0.0
6953,08B6E69DF1A8719A,20080415,20080424,-2000.0,0.0
7341,19B6ECA664FCC497,20080420,20080430,-3000.0,0.0
10270,D22793997F9DE5C3,20080525,20080603,-100.0,0.0
12374,0D1F65FFA2BF36D3,20080619,20080724,-200.0,0.0
13592,6AE60E6BB3CC83D6,20080705,20080709,-100.0,0.0
13738,BF61C24DD6FA107F,20080707,20080711,-100.0,0.0


In [ ]:
df_claims.loc[df_claims.member_id.isin(['C0230CC0BE4043F0', 'C8C35DAF0F9A864B', '6AE60E6BB3CC83D6', '8DC0BBE18AA336F6', '1E54FB03870450D2'])].sort_values('member_id')


,member_id,admission_date,discharge_date,plan_paid_amount,primary_payer_paid_amount
29642,1E54FB03870450D2,20090123,20090126,-300.0,0.0
51639,1E54FB03870450D2,20091206,20091210,6000.0,0.0
56234,1E54FB03870450D2,20100224,20100225,38000.0,0.0
13592,6AE60E6BB3CC83D6,20080705,20080709,-100.0,0.0
5312,8DC0BBE18AA336F6,20080324,20080403,4000.0,0.0
14286,8DC0BBE18AA336F6,20080714,20080718,0.0,0.0
26703,8DC0BBE18AA336F6,20081215,20081220,-300.0,0.0
11079,C0230CC0BE4043F0,20080604,20080608,4000.0,0.0
13720,C0230CC0BE4043F0,20080707,20080710,8000.0,0.0
26656,C0230CC0BE4043F0,20081214,20081222,-8000.0,0.0


Due to df_claims showing negative plan spend amount, checked to see if many negative claims exist in dataframe. Also, checked if they are claim reversals; amount & date would match another positive claim. This is not the case based on reviewing the claims of five members. Negative claims could be included for another reason, including being an oddity of a sythetic dataset.



In [ ]:
df_full = pd.merge(df_members, df_claims, on='member_id')

df_full.groupby('gender').agg(
    claim_count=('member_id', 'count'),
    total_plan_paid=('plan_paid_amount', 'sum'),
    avg_plan_paid=('plan_paid_amount', 'mean'),
    total_primary_paid=('primary_payer_paid_amount', 'sum')
)

,claim_count,total_plan_paid,avg_plan_paid,total_primary_paid
gender,,,,
Female,37751,359749510.0,9529.535906,14970600.0
Male,29022,279510670.0,9630.992695,11665100.0


Utilization by gender: matches numbers from SQL query

Initially grouped gender using df_members instead of the merged df_full. The merge creates a fresh index, and also duplicates rows since a member can have multiple claims; so the old and new indexes don't line up anymore. No error was thrown, since both tables had valid index numbers; it just matched the wrong labels to the wrong rows. Fixed by grouping on the gender column in df_full instead, since it already lines up correctly with each row

Used agg() to compute multiple aggregations in one table.

In [ ]:
df_claims.isnull().sum()

,0
member_id,0
admission_date,0
discharge_date,0
plan_paid_amount,0
primary_payer_paid_amount,0


Checked for nulls in df_claims, none present. Re-verified via SQL, no nulls currently present in created table (fact_claims) nor the original file in plan_paid_amount, primary_payer_paid_amount, NCH_PRMRY_PYR_CLM_PD_AMT nor clm_pmt_amt columns. Original Coalesce done in SQL to address nulls in plan_paid_amount may reflect an earlier data state. I've kept it there as a safeguard since it's harmless. fillna() not necessary.


In [ ]:
print(pd.to_datetime(df_full['admission_date'], format='%Y%m%d'))
print(pd.to_datetime(df_full['discharge_date'], format='%Y%m%d'))

df_full['admission_date'] = pd.to_datetime(df_full['admission_date'], format='%Y%m%d')
df_full['discharge_date'] = pd.to_datetime(df_full['discharge_date'], format='%Y%m%d')




0       2009-01-25
1       2009-02-01
2       2010-09-18
3       2007-12-27
4       2008-10-03
           ...    
66768   2009-06-23
66769   2009-08-24
66770   2010-05-14
66771   2010-07-25
66772   2009-07-28
Name: admission_date, Length: 66773, dtype: datetime64[ns]
0       2009-01-29
1       2009-02-16
2       2010-09-27
3       2008-01-04
4       2008-10-07
           ...    
66768   2009-06-27
66769   2009-08-25
66770   2010-05-26
66771   2010-08-05
66772   2009-07-31
Name: discharge_date, Length: 66773, dtype: datetime64[ns]


In [ ]:
df_full['length_of_stay'] = df_full['discharge_date'] - df_full['admission_date']
print(df_full['length_of_stay'])

0        4 days
1       15 days
2        9 days
3        8 days
4        4 days
          ...  
66768    4 days
66769    1 days
66770   12 days
66771   11 days
66772    3 days
Name: length_of_stay, Length: 66773, dtype: timedelta64[ns]


Converted dates from plain integers to a datetime type and calculated length of stay per claim.

In [ ]:
df_full.groupby('member_health_profile').agg(
    claim_count=('member_id','count'),
    total_plan_paid=('plan_paid_amount','sum'),
    avg_plan_paid=('plan_paid_amount', 'mean'),
    total_primary_paid=('primary_payer_paid_amount', 'sum')
)

,claim_count,total_plan_paid,avg_plan_paid,total_primary_paid
member_health_profile,,,,
All 3 Conditions,24749,246380980.0,9955.189301,9477900.0
Diabetes & Heart Failure,11457,101879980.0,8892.378459,4045200.0
Diabetes & Kidney Disease,3953,37958430.0,9602.436124,1894300.0
Diabetes Only,8302,77114550.0,9288.671404,3475100.0
Heart Failure Only,4378,40755680.0,9309.200548,1818500.0
Kidney & Heart Failure,2662,26387940.0,9912.824944,1287000.0
Kidney Disease Only,1111,11014000.0,9913.591359,346000.0
No Tracked Chronic Conditions,10161,97768620.0,9621.948627,4291700.0


Member Health Profile Utilization: matches results from SQL query



In [ ]:
df_full['condition_count'] = df_full['has_diabetes'] + df_full['has_kidney_disease'] + df_full['has_heart_failure']

condition_summary = df_full.groupby('condition_count').agg(
    unique_members=('member_id', 'nunique'),
    claim_count=('member_id', 'count'),
    total_plan_paid=('plan_paid_amount', 'sum'),
    avg_plan_paid_amount=('plan_paid_amount', 'mean'),
    total_primary_paid=('primary_payer_paid_amount', 'sum')
)



                 claim_count  total_plan_paid  avg_plan_paid_amount  \
condition_count                                                       
0                      10161       97768620.0           9621.948627   
1                      13791      128884230.0           9345.531869   
2                      18072      166226350.0           9198.005201   
3                      24749      246380980.0           9955.189301   

                 total_primary_paid  
condition_count                      
0                         4291700.0  
1                         5639600.0  
2                         7226500.0  
3                         9477900.0  


,condition_count,unique_members
0,0,8452
1,1,9915
2,2,10227
3,3,9186


In [ ]:
condition_summary['claims_per_member'] = condition_summary['claim_count'] / condition_summary['unique_members']
condition_summary

NameError: name 'df_full' is not defined

In [ ]:
condition_2 = df_full[df_full['condition_count'] == 2]

condition_2.groupby('member_health_profile')['plan_paid_amount'].mean().reset_index(name='avg_plan_paid')

,member_health_profile,avg_plan_paid
0,Diabetes & Heart Failure,8892.378459
1,Diabetes & Kidney Disease,9602.436124
2,Kidney & Heart Failure,9912.824944


Noted in above queries that condition count 2 has the most unique members and second highest total plan paid spent, but the lowest average plan paid amount. It has the second most total claims and claims per member, to be as expected. Checked if one of the condition combinations is pulling the average down. All three land within a 10% spread (8.9k - 9.9k). So, the pattern holds: those with 2 conditions have more frequent, but moderately lower priced visits.  

In [ ]:
df_full.groupby(['gender', 'member_health_profile']).agg(
    claim_count=('member_id', 'count'),
    total_plan_paid=('plan_paid_amount', 'sum'),
    avg_plan_paid=('plan_paid_amount', 'mean'),
    primary_payer_paid = ('primary_payer_paid_amount', 'sum')
)

claim_count  total_plan_paid  \
gender member_health_profile                                         
Female All 3 Conditions                     13829      137156570.0   
       Diabetes & Heart Failure              6627       58752620.0   
       Diabetes & Kidney Disease             2328       21936600.0   
       Diabetes Only                         4832       44989910.0   
       Heart Failure Only                    2506       23098380.0   
       Kidney & Heart Failure                1467       14552030.0   
       Kidney Disease Only                    614        6208700.0   
       No Tracked Chronic Conditions         5548       53054700.0   
Male   All 3 Conditions                     10920      109224410.0   
       Diabetes & Heart Failure              4830       43127360.0   
       Diabetes & Kidney Disease             1625       16021830.0   
       Diabetes Only                         3470       32124640.0   
       Heart Failure Only                    1872       17657300.0   
       Kidney & Heart Failure                1195       11835910.0   
       Kidney Disease Only                    497        4805300.0   
       No Tracked Chronic Conditions         4613       44713920.0   

                                      avg_plan_paid  primary_payer_paid  
gender member_health_profile                                             
Female All 3 Conditions                 9918.039627           5332600.0  
       Diabetes & Heart Failure         8865.643579           2383200.0  
       Diabetes & Kidney Disease        9422.938144            873300.0  
       Diabetes Only                    9310.825745           2137000.0  
       Heart Failure Only               9217.230646           1082500.0  
       Kidney & Heart Failure           9919.584185            497000.0  
       Kidney Disease Only             10111.889251            226000.0  
       No Tracked Chronic Conditions    9562.851478           2439000.0  
Male   All 3 Conditions                10002.235348           4145300.0  
       Diabetes & Heart Failure         8929.060041           1662000.0  
       Diabetes & Kidney Disease        9859.587692           1021000.0  
       Diabetes Only                    9257.821326           1338100.0  
       Heart Failure Only               9432.318376            736000.0  
       Kidney & Heart Failure           9904.527197            790000.0  
       Kidney Disease Only              9668.611670            120000.0  
       No Tracked Chronic Conditions    9693.024062           1852700.0

Grouping by gender and member health profile together created 16 rows, 2 genders x 8 health profiles, one row for every combination. Numbers matched the single column versions as expected. Output came back as a MultiIndex, which I liked visually.

In [ ]:
df_full.groupby('member_health_profile')['length_of_stay'].mean()

,length_of_stay
member_health_profile,
All 3 Conditions,6 days 08:43:01.049739383
Diabetes & Heart Failure,5 days 07:49:03.912018853
Diabetes & Kidney Disease,5 days 16:58:31.662028838
Diabetes Only,5 days 04:09:04.639845820
Heart Failure Only,5 days 04:14:54.655093650
Kidney & Heart Failure,6 days 03:49:21.682945154
Kidney Disease Only,5 days 13:18:24.950495049
No Tracked Chronic Conditions,5 days 03:43:37.891939769


Average length of stay came back with days plus exact hours/minutes (e.g. 6 days 08:43:01 for All 3 Conditions), instead of a plain rounded number like SQL's version. Could've used .dt.days to trim it down to whole days, but kept the extra detail since I found it useful to see. Checked against SQL; 6 days 08:43 converts to roughly 6.36 days, which matches SQL's rounded 6.4 for the same profile. Checked other profiles too.
